# Text representation with Tf-Idf and basic IR models

### Implementing a Boolean Model

A boolean model takes in a query in boolean logic and returns the documents whose features match that query. Although simplistic, it is easy to implement, and can have varying applications.

In [1]:
from collections import Counter
import math
import re
import string

# !git clone https://github.com/mayank-02/boolean-retrieval-model.git
%cd boolean-retrieval-model/
from BooleanModel import BooleanModel

from datasets import load_dataset
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
import pandas as pd
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")
nltk.download("wordnet")

/Users/leander/Documents/github/leandroviajando/ir/boolean-retrieval-model


/opt/homebrew/anaconda3/envs/ir/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[nltk_data] Downloading package punkt to /Users/leander/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/leander/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/leander/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /Users/leander/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [2]:
data =  ["Belgium is famous for chocolate, beer and waffle.",
        "Have you ever had Belgian fries?",
        "I have lived in Leuven for 4 years.",
        "I never liked chocolate on my waffle.",
        "Let's go out for some beers tonight."]

In [3]:
%mkdir -p toy_data

n=0
for line in data:
    with open (f'toy_data/text_{str(n)}.txt', "w") as outfile:
        outfile.write(line)
    n+=1

In [4]:
model = BooleanModel("./toy_data/*")

In [5]:
query = ("waffle | fries")  # Boolean logic: match documents containing the terms waffle OR fries

documents = model.query(query)

for file in documents:
  with open(f'toy_data/{file}') as f:
    print(f.read())

I never liked chocolate on my waffle.
Have you ever had Belgian fries?
Belgium is famous for chocolate, beer and waffle.


In [6]:
query = ("like | live")  # Boolean logic: match documents containing the terms like OR live

documents = model.query(query)

for file in documents:
  with open(f'toy_data/{file}') as f:
    print(f.read())

I have lived in Leuven for 4 years.
I never liked chocolate on my waffle.


### Calculating TfIdf (Term Frequency - Inverse Document Frequency)

TfIdf is a way to represent documents from frequency values based on how important a word is to a document in a collection.

**Step 1:** Preprocess the text to make it uniform, lower case, and remove punctuation so it is easy to work with.

In [7]:

def preprocess(list_of_text):
    preprocessed_text = []
    for text in list_of_text:
        text = text.translate(str.maketrans('', '', string.punctuation)) # remove punctuation
        text = text.lower() # convert to lower case
        words = word_tokenize(text) # tokenize the text
        preprocessed_text.append(words)
    return preprocessed_text

preprocessed_text = preprocess(data)

for text in preprocessed_text:
    print(text)

['belgium', 'is', 'famous', 'for', 'chocolate', 'beer', 'and', 'waffle']
['have', 'you', 'ever', 'had', 'belgian', 'fries']
['i', 'have', 'lived', 'in', 'leuven', 'for', '4', 'years']
['i', 'never', 'liked', 'chocolate', 'on', 'my', 'waffle']
['lets', 'go', 'out', 'for', 'some', 'beers', 'tonight']


**Step 2:** Calculate the term frequency for the data.
Term frequency can be calculated as the number of times a term appears in a document.

In [8]:
word_count = Counter(preprocessed_text[0])

print(word_count)

Counter({'belgium': 1, 'is': 1, 'famous': 1, 'for': 1, 'chocolate': 1, 'beer': 1, 'and': 1, 'waffle': 1})


Now for all the documents:

In [9]:
term_frequency = []
for text in preprocessed_text:
    word_count = Counter(text)
    term_frequency.append(word_count)

term_frequency = pd.DataFrame.from_dict(term_frequency, orient="columns")
term_frequency = term_frequency.fillna(0)
term_frequency

,belgium,is,famous,for,chocolate,beer,and,waffle,have,you,...,never,liked,on,my,lets,go,out,some,beers,tonight
0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,...,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0


This is quite a sparse matrix with many 0 values. Can you think of a way to meaningfully reduce the size without losing much information?

Remove the stopwords! [Stopwords](https://smltar.com/stopwords) in this case can be defined as determiners, prepositions, conjunctions and other such words that do not add any content value to the document, and so can be removed.

Let's repreat the process now without the stopwords. NLTK has a predefined list of English stopwords that we can use.

In [10]:
def preprocess(list_of_text):
    preprocessed_text = []
    for text in list_of_text:
        text = text.translate(str.maketrans('', '', string.punctuation))  # remove punctuation
        text = text.lower()  # convert to lower case
        words = word_tokenize(text)  # tokenize the text
        words = [word for word in words if word not in stopwords.words('english')]  # remove stopwords
        preprocessed_text.append(words)
    return preprocessed_text

preprocessed_text = preprocess(data)
term_frequency = []
for text in preprocessed_text:
    word_count = Counter(text)
    term_frequency.append(word_count)

term_frequency = pd.DataFrame.from_dict(term_frequency, orient="columns")
term_frequency = term_frequency.fillna(0)
term_frequency

,belgium,famous,chocolate,beer,waffle,ever,belgian,fries,lived,leuven,4,years,never,liked,lets,go,beers,tonight
0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0


That's better!

**Step 3:** We can now use the term frequencies to calculate the inverse document frequency. Inverse document frequency is the log of the number of documents / number of documents containing the term.

In [11]:
idf = {}
for word in term_frequency.columns:
    idf[word] = math.log(len(preprocessed_text)/term_frequency[word].apply(lambda x: 1 if x > 0 else 0).sum())

idf = pd.DataFrame.from_dict(idf, orient="index").transpose()
idf

,belgium,famous,chocolate,beer,waffle,ever,belgian,fries,lived,leuven,4,years,never,liked,lets,go,beers,tonight
0,1.609438,1.609438,0.916291,1.609438,0.916291,1.609438,1.609438,1.609438,1.609438,1.609438,1.609438,1.609438,1.609438,1.609438,1.609438,1.609438,1.609438,1.609438


**Step 4:** Calculate the tf-idf. Tf-idf is the product of the term frequency and the inverse document frequency.

In [12]:
tfidf_dict = {}
for i in range(len(preprocessed_text)):
    document_tfidf = []
    for column in idf:
        tfidf = term_frequency[column][i] * idf[column]
        document_tfidf.append(tfidf.values[0])
    tfidf_dict[f"Document {str(i)}"] = document_tfidf

tfidf_df = pd.DataFrame.from_dict(tfidf_dict, orient="index", columns = idf.columns)
tfidf_df = tfidf_df.round(2)
tfidf_df

,belgium,famous,chocolate,beer,waffle,ever,belgian,fries,lived,leuven,4,years,never,liked,lets,go,beers,tonight
Document 0,1.61,1.61,0.92,1.61,0.92,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
Document 1,0.00,0.00,0.00,0.00,0.00,1.61,1.61,1.61,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
Document 2,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.61,1.61,1.61,1.61,0.00,0.00,0.00,0.00,0.00,0.00
Document 3,0.00,0.00,0.92,0.00,0.92,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.61,1.61,0.00,0.00,0.00,0.00
Document 4,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.61,1.61,1.61,1.61


This pipeline we created has already been implemented in sklearn, so we can just use that!

In [13]:
vectorizer = TfidfVectorizer(stop_words="english")

In [14]:
analyze = vectorizer.build_analyzer()

In [15]:
print("Document 1", analyze(data[0]))
print("Document 2", analyze(data[1]))

Document 1 ['belgium', 'famous', 'chocolate', 'beer', 'waffle']
Document 2 ['belgian', 'fries']


In [16]:
X = vectorizer.fit_transform(data)
tfidf_df = pd.DataFrame(X.toarray(), index=range(len(data)), columns=vectorizer.get_feature_names_out())
tfidf_df = tfidf_df.round(2)
tfidf_df

,beer,beers,belgian,belgium,chocolate,famous,fries,let,leuven,liked,lived,tonight,waffle,years
0,0.48,0.00,0.00,0.48,0.39,0.48,0.00,0.00,0.00,0.00,0.00,0.00,0.39,0.00
1,0.00,0.00,0.71,0.00,0.00,0.00,0.71,0.00,0.00,0.00,0.00,0.00,0.00,0.00
2,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.58,0.00,0.58,0.00,0.00,0.58
3,0.00,0.00,0.00,0.00,0.53,0.00,0.00,0.00,0.00,0.66,0.00,0.00,0.53,0.00
4,0.00,0.58,0.00,0.00,0.00,0.00,0.00,0.58,0.00,0.00,0.00,0.58,0.00,0.00


Do you note a difference in the tfidf values between the two matrices? This is because the sklearn implementation applies a **normalisation** to the `tf` (along with some **smoothing**), similar to what you have seen in class with length normalisation and augmented normalised term frequency.

How does that help?

This is to _avoid longer documents having higher tfidf weights across the vocabulary, since term frequency is sensitive to the length of the document_.

This matrix contains less words than the one we calculated manually. This is because the list of stop words defined in the nltk library is less exhaustive than the list defined in the sklearn implementation.

The size of this matrix can be reduced even further by lemmatizing the words. Lemmatizing is the process of converting words into their root form (beers --> beer). That way, when we search for 'beer', documents containing both 'beer' and 'beers' will be returned.

To do this, we can handle the entire preprocessing step ourselves, and pass the function we defined earlier to the vectorizer, after adding a lemmatizer.


In [17]:
def preprocess(doc):  # this takes in a string and converts it into a list
    doc = doc.split()
    lemmatizer = WordNetLemmatizer()
    preprocessed_text = []
    for text in doc:
        text = text.translate(str.maketrans("", "", string.punctuation))  # remove punctuation
        text = text.lower()  # convert to lower case
        words = word_tokenize(text)  # tokenize the text
        words = [word for word in words if word not in stopwords.words("english")]  # remove stopwords
        words = [lemmatizer.lemmatize(word) for word in words]  # lemmatize
        if words != []:
            preprocessed_text.append(words[0])
    return preprocessed_text

vectorizer = TfidfVectorizer(tokenizer=preprocess)
analyze = vectorizer.build_analyzer()
for idx, doc in enumerate(data):
    print(f"Document {idx}", analyze(doc))

Document 0 ['belgium', 'famous', 'chocolate', 'beer', 'waffle']
Document 1 ['ever', 'belgian', 'fry']
Document 2 ['lived', 'leuven', '4', 'year']
Document 3 ['never', 'liked', 'chocolate', 'waffle']
Document 4 ['let', 'go', 'beer', 'tonight']


In [18]:
X = vectorizer.fit_transform(data)
tfidf_df = pd.DataFrame(X.toarray(), index=range(len(data)), columns=vectorizer.get_feature_names_out())
tfidf_df = tfidf_df.round(2)
tfidf_df

/opt/homebrew/anaconda3/envs/ir/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


,4,beer,belgian,belgium,chocolate,ever,famous,fry,go,let,leuven,liked,lived,never,tonight,waffle,year
0,0.0,0.41,0.00,0.5,0.41,0.00,0.5,0.00,0.00,0.00,0.0,0.00,0.0,0.00,0.00,0.41,0.0
1,0.0,0.00,0.58,0.0,0.00,0.58,0.0,0.58,0.00,0.00,0.0,0.00,0.0,0.00,0.00,0.00,0.0
2,0.5,0.00,0.00,0.0,0.00,0.00,0.0,0.00,0.00,0.00,0.5,0.00,0.5,0.00,0.00,0.00,0.5
3,0.0,0.00,0.00,0.0,0.44,0.00,0.0,0.00,0.00,0.00,0.0,0.55,0.0,0.55,0.00,0.44,0.0
4,0.0,0.42,0.00,0.0,0.00,0.00,0.0,0.00,0.52,0.52,0.0,0.00,0.0,0.00,0.52,0.00,0.0


### Implementing a Vector Space Model

A vector space model takes a vector representation of the document and the query and ranks documents according to a similarity or distance measure between the query and document vector.

We will use the tf-idf weights we calculated above for our vector space model. The query can be represented as a simple binary vector where 1 represents the presence of a word and 0 represents the absense.

In [19]:
query = "beer and fries"
query = preprocess(query)  # preprocess the query before converting it into a vector

print(query)

['beer', 'fry']


We can now convert the query into a vector by passing it to the same vectorizer object we defined earlier:

In [20]:
# The vectorizer takes as input a list of queries - so here it will take ["beer fry"] as query.
# You can pass multiple queries as ["beer fry", "waffle fry"] which will return one vector for each query.
query_vector = vectorizer.transform([" ".join(query)])

print(query_vector)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 2 stored elements and shape (1, 17)>
  Coords	Values
  (0, 1)	0.6279137616509933
  (0, 7)	0.7782829228046183


We will use cosine similarity to measure the similarity between the query and the documents, and rank the documents accordingly.

We pass the query vector and the tf-idf matrix, internally the `cosine_similarity` function will calculate the similarity between the query vector and each document vector and return scores.


In [21]:
cosine_similarities = cosine_similarity(query_vector, X)

results = [(data[i], cosine_similarities[0][i]) for i in range(len(data))]
results.sort(key=lambda x: x[1], reverse=True)

for doc, similarity in results:
    print(f"Similarity: {similarity:.2f}\n{doc}\n")

Similarity: 0.45
Have you ever had Belgian fries?

Similarity: 0.27
Let's go out for some beers tonight.

Similarity: 0.25
Belgium is famous for chocolate, beer and waffle.

Similarity: 0.00
I have lived in Leuven for 4 years.

Similarity: 0.00
I never liked chocolate on my waffle.



**Task:** Sklearn has implementations for more similarity and distance metrics such as Euclidean distance and Manhattan distance. Try using another metric and compare the difference in results.

In [22]:
from sklearn.metrics.pairwise import cosine_distances

cosine_distances_similarities = cosine_distances(query_vector, X)

results = [(data[i], cosine_distances_similarities[0][i]) for i in range(len(data))]
results.sort(key=lambda x: x[1], reverse=True)

for doc, similarity in results:
    print(f"Similarity: {similarity:.2f}\n{doc}\n")

Similarity: 1.00
I have lived in Leuven for 4 years.

Similarity: 1.00
I never liked chocolate on my waffle.

Similarity: 0.75
Belgium is famous for chocolate, beer and waffle.

Similarity: 0.73
Let's go out for some beers tonight.

Similarity: 0.55
Have you ever had Belgian fries?



In [23]:
from sklearn.metrics.pairwise import manhattan_distances  # L1 distance

manhattan_distances_similarities = manhattan_distances(query_vector, X)

results = [(data[i], manhattan_distances_similarities[0][i]) for i in range(len(data))]
results.sort(key=lambda x: x[1], reverse=True)

for doc, similarity in results:
    print(f"Similarity: {similarity:.2f}\n{doc}\n")

Similarity: 3.41
I have lived in Leuven for 4 years.

Similarity: 3.39
I never liked chocolate on my waffle.

Similarity: 2.82
Belgium is famous for chocolate, beer and waffle.

Similarity: 2.55
Let's go out for some beers tonight.

Similarity: 1.98
Have you ever had Belgian fries?



In [24]:
from sklearn.metrics.pairwise import euclidean_distances

euclidean_distances_similarities = euclidean_distances(query_vector, X)  # L2 distance

results = [(data[i], euclidean_distances_similarities[0][i]) for i in range(len(data))]
results.sort(key=lambda x: x[1], reverse=True)

for doc, similarity in results:
    print(f"Similarity: {similarity:.2f}\n{doc}\n")

Similarity: 1.41
I have lived in Leuven for 4 years.

Similarity: 1.41
I never liked chocolate on my waffle.

Similarity: 1.22
Belgium is famous for chocolate, beer and waffle.

Similarity: 1.21
Let's go out for some beers tonight.

Similarity: 1.05
Have you ever had Belgian fries?



In [25]:
from sklearn.metrics.pairwise import haversine_distances

haversine_distances_similarities = haversine_distances(query_vector, X)

results = [(data[i], haversine_distances_similarities[0][i]) for i in range(len(data))]
results.sort(key=lambda x: x[1], reverse=True)

for doc, similarity in results:
    print(f"Similarity: {similarity:.2f}\n{doc}\n")

Similarity: 0.61
Have you ever had Belgian fries?

Similarity: 0.57
I have lived in Leuven for 4 years.

Similarity: 0.37
Belgium is famous for chocolate, beer and waffle.

Similarity: 0.33
I never liked chocolate on my waffle.

Similarity: 0.25
Let's go out for some beers tonight.



### Implementing BM25 (Probabilistic model)

The Best-Match 25 algorithm follows the probabilistic retrieval framework, and uses term frequency and document length normalisation to determine the relevance of a document given a query. It operates with the underying assumption that a document generates a query.

The BM25 score for a document D with respect to a query Q is calculated as the sum of the scores for individual query terms. The formula for calculating the BM25 score is as follows:

$$BM25(D, Q) = ∑(IDF(q) * ((TF(q, D) * (k1 + 1)) / (TF(q, D) + k1 * (1 — b + b * (|D| / avgdl)))))$$

In this formula, $IDF(q)$ represents the inverse document frequency of the query term $q$, $TF(q, D)$ denotes the modified term frequency of term $q$ in document $D$, $\lvert D \rvert$ represents the length of document $D$, and $avgdl$ is the average document length in the corpus. Parameters $k1$ and $b$ are tunable constants that control the impact of term frequency saturation and document length normalization, respectively.

Note that BM25 measures TF and IDF differently. BM25 uses a modified term frequency that takes into account saturation effects to prevent overemphasizing heavily repeated terms. For IDF it assigns higher weights to terms that are rare in the corpus and lower weights to terms that are common.

Following is an implementation of the Okapi BM25 algorithm, taken from [this paper](http://www.cs.otago.ac.nz/homepages/andrew/papers/2014-2.pdf). To understand the implementation, see https://github.com/dorianbrown/rank_bm25/tree/master

Before we move ahead, we have to preprocess the data again since we added the lemmatizer to our preprocessing function.

In [26]:
preprocessed_text = [preprocess(doc) for doc in data]

In [27]:
bm25 = BM25Okapi(preprocessed_text)  # fit the model to the data

Let's define our query again:

In [28]:
query = "beer and fries"
query = preprocess(query)

We pass the preprocessed query and data to the model and rank documents accordingly:

In [29]:
# this returns an array of scores, where we can map the index to the index of the data to get the ranking
scores = bm25.get_scores(query)

score_to_id = {}
for i in range(len(scores)):
    score_to_id[str(i)] = float(scores[i])

ranking = sorted(score_to_id.items(), key=lambda x:x[1], reverse=True)

for k,v in ranking[:5]:
    print(f"Document: {data[int(k)]}\n Score: {v}")

Document: Have you ever had Belgian fries?
 Score: 1.23787300131618
Document: Let's go out for some beers tonight.
 Score: 0.33647223662121295
Document: Belgium is famous for chocolate, beer and waffle.
 Score: 0.30244695426625884
Document: I have lived in Leuven for 4 years.
 Score: 0.0
Document: I never liked chocolate on my waffle.
 Score: 0.0


**Task**: Try different queries.

In [30]:
# this returns an array of scores, where we can map the index to the index of the data to get the ranking
scores = bm25.get_scores(preprocess("Belgium"))

score_to_id = {}
for i in range(len(scores)):
    score_to_id[str(i)] = float(scores[i])

ranking = sorted(score_to_id.items(), key=lambda x:x[1], reverse=True)

for k,v in ranking[:5]:
    print(f"Document: {data[int(k)]}\n Score: {v}")

Document: Belgium is famous for chocolate, beer and waffle.
 Score: 0.9875166639713346
Document: Have you ever had Belgian fries?
 Score: 0.0
Document: I have lived in Leuven for 4 years.
 Score: 0.0
Document: I never liked chocolate on my waffle.
 Score: 0.0
Document: Let's go out for some beers tonight.
 Score: 0.0


### Exercise: Load your own data

For this exercise, you will load a subset of the [Simple English Wikipedia](https://simple.wikipedia.org/wiki/Main_Page), which is small and easy to load.

**Part 1**: Load the dataset and implement a boolean model, a vector space model
and a probabilistic model on the dataset for the following queries:

1. Earth's atmosphere
2. Agricultural crops
3. Parts of the human body
4. What are the official languages of countries?
5. Best places to travel

**Part 2**: Rank the documents and compare the top 5 results for each model. What differences do you see? Try passing longer queries to the models, for example, "Name parts of the human body" instead of "parts of the human body". Does that change the result?

**Part 3**: What happens to the results if you don't lemmatize or remove stop words?

**Part 4**: Try a different similarity measure for the vector space model. Which one returns more suitable results?

**Extra:**

- The BM25 repository provided above contains different variations of the BM25 algorithm - BM25L and BM25Plus. Use these models on the dataset and rank documents based on the queries defined. Compare the results.
- The hyperparameters k1 and b in the BM25 algorithm can be tuned. Test a couple of different values - can it be optimized further?

#### Dataset:

The Simple English Wikipedia can be loaded using the huggingface datasets library. We will only work with 200 articles from the dataset to save on computation time.

Note: You can increase the size of the subset to observe differences in the ranking of the retrieved documents.

In [31]:
data = load_dataset("wikipedia", "20220301.simple", trust_remote_code=True)["train"]
data = data.select(range(200))

In [32]:
print(data["text"][0])

April is the fourth month of the year in the Julian and Gregorian calendars, and comes between March and May. It is one of four months to have 30 days.

April always begins on the same day of week as July, and additionally, January in leap years. April always ends on the same day of the week as December.

April's flowers are the Sweet Pea and Daisy. Its birthstone is the diamond. The meaning of the diamond is innocence.

The Month 

April comes between March and May, making it the fourth month of the year. It also comes first in the year out of the four months that have 30 days, as June, September and November are later in the year.

April begins on the same day of the week as July every year and on the same day of the week as January in leap years. April ends on the same day of the week as December every year, as each other's last days are exactly 35 weeks (245 days) apart.

In common years, April starts on the same day of the week as October of the previous year, and in leap years, M

In [33]:
# The implementation requies data in the form of text files, so let's create them.
# We will use the data ids provided for each article in the dataset as filenames so we can retrieve them later.

%mkdir data
for doc in data:
  with open (f"data/{doc['id']}.txt", "w") as outfile:
    outfile.write(doc["text"])

mkdir: data: File exists


In [34]:
model = BooleanModel("./data/*")

# Boolean logic
for query in [("earth & atmosphere"), ("agricultural & crops"), ("parts & human & body"),
              ("country & official & language"), ("travel & best & tourism")]:
    results = model.query(query)
    print(results)

['262.txt', '48.txt', '9.txt', '355.txt', '219.txt', '218.txt', '243.txt', '296.txt']
['102.txt', '80.txt', '280.txt', '294.txt', '19.txt']
['112.txt', '305.txt', '48.txt', '57.txt', '209.txt', '353.txt', '50.txt', '243.txt', '242.txt', '286.txt']
['262.txt', '103.txt', '248.txt', '12.txt', '104.txt', '89.txt', '361.txt', '363.txt', '208.txt', '54.txt', '178.txt', '52.txt', '185.txt', '53.txt', '292.txt', '27.txt', '291.txt']
['178.txt']


In [35]:
regex = re.compile(r'\d+')

result_ids = []
for result in results:
  id = regex.findall(result)[0]  # get the data ids from the filenames in the results
  result_ids.append(id)

for article in data:
  if article["id"] in result_ids:  # use the ids to get those datapoints from the dataset object
    print(f"{article}\n")

{'id': '178', 'url': 'https://simple.wikipedia.org/wiki/Cuba', 'title': 'Cuba', 'text': 'Cuba is an island country in the Caribbean Sea. The country is made up of the big island of Cuba, the Isla de la Juventud island (Isle of Youth), and many smaller islands. Havana is the capital of Cuba. It is the largest city. The second largest city is Santiago de Cuba. In Spanish, the capital is called "La Habana". Cuba is near the United States, Mexico, Haiti, Jamaica and the Bahamas. People from Cuba are called Cubans (cubanos in Spanish). The official language is Spanish. Cuba is warm all year.\n\nIn 1492, Christopher Columbus landed on the island of Cuba. He claimed it for the Kingdom of Spain. Cuba became a Spanish colony until the Spanish–American War of 1898. After the war, it was part of the United States. It gained independence in 1902.\n\nIn 1959, guerrilla fighters led by Fidel Castro and Che Guevara overthrew Cuba\'s dictator, Fulgencio Batista, in what became the Cuban Revolution. Ca

This type of model can be fast and efficient, since it is just doing literal string matching. For applications where you just want to do a keyword search, for example, this is very good.

Moreover, it lets you also define words that you do not want to appear in the documents, which is not possible in the vector space or probabilistic models.

On the other hand, you have to be able to fit your queries into the structure of boolean logic. This becomes difficult for longer, more complex queries.

**Implementing a Vector Space Model**:

A vector space model takes a vector representation of the document and the query, and ranks documents according to a similarity or distance measure between the query and document vector.

We will use the tf-idf weights we calculated above for our vector space model. The query can be represented as a simple binary vector where 1 represents the presence of a word and 0 represents its absense. In this implementation, the query vector is basically a vector of the size of the vocabulary of the corpus, with a tfidf weight assigned to the words present in the query.

In [36]:
from sklearn.feature_extraction.text import TfidfVectorizer


def preprocess(doc: str) -> list:
    doc = doc.split()
    digits = re.compile(r'\d')
    lemmatizer = WordNetLemmatizer()
    preprocessed_text = []
    for text in doc:
        text = text.translate(str.maketrans("", "", string.punctuation))  # remove punctuation
        text = text.lower()  # convert to lower case
        words = word_tokenize(text)  # tokenize the text
        words = [word for word in words if word not in stopwords.words("english")]  # remove stopwords
        words = [word for word in words if not digits.match(word)]  #  additional step : removing digits
        words = [lemmatizer.lemmatize(word) for word in words]  # lemmatize
        if words != []:
            preprocessed_text.append(words[0])
    return preprocessed_text

vectorizer = TfidfVectorizer(tokenizer=preprocess)
X = vectorizer.fit_transform(data["text"])
tfidf_df = pd.DataFrame(X.toarray(), index=range(len(data)), columns=vectorizer.get_feature_names_out())
tfidf_df = tfidf_df.round(2)
tfidf_df

/opt/homebrew/anaconda3/envs/ir/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


,aaa,aalto,aave,abacus,abalone,abandoned,abbasids,abbey,abbott,abbreviated,...,€725,→,−,≠,八百万の神,大都,姬,市,燕,蓟
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
196,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
197,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.02,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
198,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [37]:
query = "Earth's atmosphere"
query = "Agricultural crops"
query = "Parts of the human body"
query = "What are the official languages of countries?"
query = "name parts of the human body."
query = "Best places to travel as a tourist"

query = preprocess(query)
print(query)

['best', 'place', 'travel', 'tourist']


In [38]:
query_vector = vectorizer.transform([" ".join(query)])
print(query_vector)
print(query_vector.toarray())  # this is the actual vector

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4 stored elements and shape (1, 11033)>
  Coords	Values
  (0, 1051)	0.5079227478218901
  (0, 7521)	0.333789303173216
  (0, 10061)	0.595398887797236
  (0, 10119)	0.5254515655529196
[[0. 0. 0. ... 0. 0. 0.]]


In [39]:
from sklearn.metrics.pairwise import cosine_similarity

cosine_similarities = cosine_similarity(query_vector, X)
results = [(data[i], cosine_similarities[0][i]) for i in range(len(data))]
results.sort(key=lambda x: x[1], reverse=True)
for doc, similarity in results[:5]:
    print(f"Similarity: {similarity:.2f}\n{doc}\n")

Similarity: 0.05
{'id': '182', 'url': 'https://simple.wikipedia.org/wiki/Cost%20of%20living', 'title': 'Cost of living', 'text': 'Cost of living is the amount of money it costs just to live in a certain place.  It includes food, housing, etc.\n\nEconomics'}

Similarity: 0.05
{'id': '186', 'url': 'https://simple.wikipedia.org/wiki/Dublin', 'title': 'Dublin', 'text': 'Dublin () is the capital of the Republic of Ireland, and the biggest city on the island of Ireland. In 2011, there were over 1.1 million people living in the Greater Dublin Area.\n\nDublin was built by the Vikings upon the river Liffey. The river divides the city into two parts, North Dublin and South Dublin.\n\nMany famous writers lived in Dublin. Oscar Wilde and George Bernard Shaw were born in Dublin. James Joyce is probably Dublin\'s best known and most international writer.\n\nDublin is home to Ireland\'s largest stadium for all sports, Croke Park. It can hold up to 85,000 people. Croke Park is the usual venue for all 

In [40]:
from sklearn.metrics.pairwise import euclidean_distances

euclidean_dist = euclidean_distances(query_vector, X)
results = [(data[i], euclidean_dist[0][i]) for i in range(len(data))]
results.sort(key=lambda x: x[1])
for doc, similarity in results[:5]:
    print(f"Distance: {similarity:.2f}\n{doc}\n")

Distance: 1.38
{'id': '182', 'url': 'https://simple.wikipedia.org/wiki/Cost%20of%20living', 'title': 'Cost of living', 'text': 'Cost of living is the amount of money it costs just to live in a certain place.  It includes food, housing, etc.\n\nEconomics'}

Distance: 1.38
{'id': '186', 'url': 'https://simple.wikipedia.org/wiki/Dublin', 'title': 'Dublin', 'text': 'Dublin () is the capital of the Republic of Ireland, and the biggest city on the island of Ireland. In 2011, there were over 1.1 million people living in the Greater Dublin Area.\n\nDublin was built by the Vikings upon the river Liffey. The river divides the city into two parts, North Dublin and South Dublin.\n\nMany famous writers lived in Dublin. Oscar Wilde and George Bernard Shaw were born in Dublin. James Joyce is probably Dublin\'s best known and most international writer.\n\nDublin is home to Ireland\'s largest stadium for all sports, Croke Park. It can hold up to 85,000 people. Croke Park is the usual venue for all Irel

Both similarity metrics return the same rankings.

Okapi BM25 (Probabilistic model)

In [41]:
preprocessed_text = [preprocess(doc) for doc in data["text"]]

In [42]:
bm25 = BM25Okapi(preprocessed_text)

In [43]:
query = "Earth's atmosphere"
query = "Agricultural crops"
query = "Parts of the human body"
query = "What are the official languages of countries?"
query = "name parts of the human body."
query = "Best places to travel as a tourist"

query = preprocess(query)
print(query)

['best', 'place', 'travel', 'tourist']


In [44]:
doc_scores = bm25.get_scores(query)

score_to_id = {}
for i in range(len(doc_scores)):
  score_to_id[str(i)] = float(doc_scores[i])

ranking = sorted(score_to_id.items(), key=lambda x:x[1], reverse=True)

for k,v in ranking[:5]:
  print(f"Document: {data[int(k)]}\n Score: {v}\n")

Document: {'id': '186', 'url': 'https://simple.wikipedia.org/wiki/Dublin', 'title': 'Dublin', 'text': 'Dublin () is the capital of the Republic of Ireland, and the biggest city on the island of Ireland. In 2011, there were over 1.1 million people living in the Greater Dublin Area.\n\nDublin was built by the Vikings upon the river Liffey. The river divides the city into two parts, North Dublin and South Dublin.\n\nMany famous writers lived in Dublin. Oscar Wilde and George Bernard Shaw were born in Dublin. James Joyce is probably Dublin\'s best known and most international writer.\n\nDublin is home to Ireland\'s largest stadium for all sports, Croke Park. It can hold up to 85,000 people. Croke Park is the usual venue for all Ireland hurling and football finals. The Aviva Stadium hosts rugby and soccer.\n\nNotes\n\nReferences\n\nOther websites \n\n "The Reflecting City"\n Dublin GDP stats \n WikiSatellite view of Dublin at WikiMapia\n The Dublin Community Blog\n Satellite map of Dublin -

While the top 3 or 4 ranked documents are the same for both vector space and probabilistic models (they both use tfidf, although with different variations) the differences in the models come out when we go down the rankings.

Although just based on word frequencies, tfidf can be a powerful representation that is able to identify relevance in a meaningful manner.

**What happens to the tfidf matrix if you don't preprocess the text? How does that affect the results from your vector space model?**

In [45]:
def preprocess(doc):   #this takes in a string and converts it into a list
    doc = doc.split()
    # preprocessed_text = []
    # for text in doc:
    #     text = text.translate(str.maketrans('', '', string.punctuation)) #remove punctuation
    #     text = text.lower() #convert to lower case
    #     words = word_tokenize(text) #tokenize the text
    #     words = [word for word in words if word not in stopwords.words('english')] #remove stopwords
    #     words = [lemmatizer.lemmatize(word) for word in words] #lemmatize
    #     if words != []:
    #         preprocessed_text.append(words[0])
    return doc

vectorizer = TfidfVectorizer(tokenizer=preprocess)
X = vectorizer.fit_transform(data['text'])
tfidf_df = pd.DataFrame(X.toarray(), index=range(len(data)), columns=vectorizer.get_feature_names_out())
tfidf_df = tfidf_df.round(2)
tfidf_df

/opt/homebrew/anaconda3/envs/ir/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


,"""","""""",""""",","""&"",","""&c""","""+""","""+"",",""".","""...knödel"";","""...nudln""",...,€2.64,€23.5,€3.1,"€451,000",€56,€7.25,→,−,≠,市
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
196,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.06,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
197,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
198,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


You will observe that there is a lot of noise retained in the matrix that does not add any value to the representations.

You are essentially storing noise and adding compute time, and assigning weights to words/characters that are irrelevant.

You can try running the queries again to see how that impacts the type of documents your model will return.

In [46]:
query = "Earth's atmosphere"
query = "Agricultural crops"
query = "Parts of the human body"
query = "What are the official languages of countries?"
query = "name parts of the human body."
query = "Best places to travel as a tourist"

query = preprocess(query)
print(query)

['Best', 'places', 'to', 'travel', 'as', 'a', 'tourist']


The query also retains the stop words - so you will be computing similarities for 'of' and 'the' that add no additional information to the meaning of your query - they just serve a grammatical purpose.

Vector Space Model

In [47]:
query_vector = vectorizer.transform([" ".join(query)])
print(query_vector)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 7 stored elements and shape (1, 20656)>
  Coords	Values
  (0, 2524)	0.12939466803678873
  (0, 3598)	0.16237442934105703
  (0, 4259)	0.43364021029270233
  (0, 14950)	0.43364021029270233
  (0, 18959)	0.1360081192592064
  (0, 19073)	0.5865693839850251
  (0, 19157)	0.46716383623030877


In [48]:
cosine_similarities = cosine_similarity(query_vector, X)
results = [(data[i], cosine_similarities[0][i]) for i in range(len(data))]
results.sort(key=lambda x: x[1], reverse=True)
for doc, similarity in results[:5]:
    print(f"Similarity: {similarity:.2f}\n{doc}\n")

Similarity: 0.10
{'id': '144', 'url': 'https://simple.wikipedia.org/wiki/City', 'title': 'City', 'text': 'A city is a heavily inhabited community that may include structures, buildings, bridges, rivers or lakes, and landmarks.\n\nA city has many buildings and streets. It has houses, hotels, condominiums, and apartments for many people to live in, shops where they may buy things, places for people to work, and a government to run the city and keep law and order in the city. People live in cities because it is easy for them to find and do everything they want there. A city usually has a "city center" where government and business occur and suburbs where people live outside the center.\n\nDefinition\n\nNo rule is used worldwide to decide why some places are called "city," and other places are called "town."\n\nSome things that make a city are :\n A long history. Although many cities today have only been around for tens or hundreds of years, there are a few which have been so for thousands

In [49]:
euclidean_dist = euclidean_distances(query_vector, X)
results = [(data[i], euclidean_dist[0][i]) for i in range(len(data))]
results.sort(key=lambda x: x[1])
for doc, similarity in results[:5]:
    print(f"Distance: {similarity:.2f}\n{doc}\n")

Distance: 1.34
{'id': '144', 'url': 'https://simple.wikipedia.org/wiki/City', 'title': 'City', 'text': 'A city is a heavily inhabited community that may include structures, buildings, bridges, rivers or lakes, and landmarks.\n\nA city has many buildings and streets. It has houses, hotels, condominiums, and apartments for many people to live in, shops where they may buy things, places for people to work, and a government to run the city and keep law and order in the city. People live in cities because it is easy for them to find and do everything they want there. A city usually has a "city center" where government and business occur and suburbs where people live outside the center.\n\nDefinition\n\nNo rule is used worldwide to decide why some places are called "city," and other places are called "town."\n\nSome things that make a city are :\n A long history. Although many cities today have only been around for tens or hundreds of years, there are a few which have been so for thousands o

Probabilistic Model

In [50]:
preprocessed_text = [preprocess(doc) for doc in data['text']]

In [51]:
bm25 = BM25Okapi(preprocessed_text)

In [52]:
doc_scores = bm25.get_scores(query)

score_to_id = {}
for i in range(len(doc_scores)):
    score_to_id[str(i)] = float(doc_scores[i])

ranking = sorted(score_to_id.items(), key=lambda x:x[1], reverse=True)

for k,v in ranking[:5]:
    print(f'Document: {data[int(k)]}\n Score: {v}\n')

Document: {'id': '144', 'url': 'https://simple.wikipedia.org/wiki/City', 'title': 'City', 'text': 'A city is a heavily inhabited community that may include structures, buildings, bridges, rivers or lakes, and landmarks.\n\nA city has many buildings and streets. It has houses, hotels, condominiums, and apartments for many people to live in, shops where they may buy things, places for people to work, and a government to run the city and keep law and order in the city. People live in cities because it is easy for them to find and do everything they want there. A city usually has a "city center" where government and business occur and suburbs where people live outside the center.\n\nDefinition\n\nNo rule is used worldwide to decide why some places are called "city," and other places are called "town."\n\nSome things that make a city are :\n A long history. Although many cities today have only been around for tens or hundreds of years, there are a few which have been so for thousands of yea

### Sources

- https://medium.com/@ramishamukhtar786/boolean-retrieval-model-be5843ae3c9e
- https://github.com/mayank-02/boolean-retrieval-model/tree/main
- https://www.imaurer.com/which-vector-similarity-metric-should-i-use/
- https://spotintelligence.com/2023/09/07/vector-space-model/
- https://medium.com/analytics-vidhya/tf-idf-term-frequency-technique-easiest-explanation-for-text-classification-in-nlp-with-code-8ca3912e58c3
- https://medium.com/@rebirth4vali/tf-idf-from-scratch-with-python-e22033bb99f
- https://melaniewalsh.github.io/Intro-Cultural-Analytics/05-Text-Analysis/03-TF-IDF-Scikit-Learn.html
- https://medium.com/@evertongomede/understanding-the-bm25-ranking-algorithm-19f6d45c6ce